## Baseline Logistic Regression Model

This notebook serves to test the following features:

- The missing transverse energy (<i>PRI_met</i>)
- The total missing transverse momentum (<i>DER_pt_tot</i>)
- (<i>DER_mass_jet_jet</i>)
- (<i>DER_pt_h</i>)

These four features were selected due to the separation of signal and noise in 02_feature_investigation.

### Import Libraries and Modules

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance

root = Path().resolve()
while not (root/"src").exists():
    root = root.parent
sys.path.append(str(root))

from src.data_processing import pull_data, create_target, select_features
from src.training_functions import splitting_data, scaling_data
from src.evaluation_metrics import evaluate_model

### Process Data

For this model, only creating a binary target for signals is necessary.

In [2]:
df = pull_data("data/atlas-higgs-challenge-2014-v2.csv")
df = create_target(df)

### Select Training and Test Data

In [3]:
x_data, y_data = select_features(df, ["PRI_met", "DER_pt_tot", "DER_mass_jet_jet", "DER_pt_h"], "Target")

x_train, x_test, y_train, y_test = splitting_data(x_data, y_data)
x_train_scaled, x_test_scaled, scaler = scaling_data(x_train, x_test)

### Create the Linear Regression

In [4]:
model = LogisticRegression()
model.fit(x_train_scaled, y_train)
print(model.score(x_test_scaled, y_test))

0.682513687915526


### Attain the Prediction and Probability Values from the Model

In [5]:
predictions = model.predict(x_test_scaled)
probability = model.predict_proba(x_test_scaled)[:,1]

### Evaluation of Accuracy

In [6]:
evaluation_df = evaluate_model(y_test, predictions, probability)
print(evaluation_df)

          0         1
0  Accuracy  0.682514
1   roc-auc  0.678248
2  Log Loss  0.603640


### Importance of the Selected Features

In [7]:
coefficients = model.coef_[0]
odds_ratios = np.exp(coefficients)
perm_importance = permutation_importance(model, x_test_scaled, y_test)

feature_importance = pd.DataFrame({
    "Feature": x_test.columns,
    "Coefficient": coefficients,
    "Odds Ratio": odds_ratios,
    "Importance Mean": perm_importance.importances_mean,
    "Importance STD": perm_importance.importances_std
})

print(feature_importance.sort_values(by = "Coefficient", ascending=False))

            Feature  Coefficient  Odds Ratio  Importance Mean  Importance STD
3          DER_pt_h     0.638801    1.894207         0.053265        0.000318
2  DER_mass_jet_jet     0.256987    1.293028         0.018790        0.000540
1        DER_pt_tot    -0.199088    0.819478         0.011806        0.000239
0           PRI_met    -0.444788    0.640960         0.007813        0.000293
